In [1]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

FINAL_DIR = "../Models/medha_final_model"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)

# Create model architecture
medha_model = AutoModelForSequenceClassification.from_pretrained(
    "google/muril-base-cased",
    num_labels=5
)

# Load your trained weights
state_dict = torch.load(
    os.path.join(FINAL_DIR, "medha_model.pt"),
    map_location=device
)

medha_model.load_state_dict(state_dict)
medha_model.to(device)
medha_model.eval()

# Load config
with open(os.path.join(FINAL_DIR, "medha_config.json"), "r") as f:
    medha_config = json.load(f)

LABELS = medha_config["labels"]
THRESHOLDS = medha_config["thresholds"]

print("Model loaded successfully.")
print("Device:", device)
print("Labels:", LABELS)
print("Thresholds:", THRESHOLDS)

c:\Users\jnark\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14198.87it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNor

Model loaded successfully.
Device: cuda
Labels: ['Distress_Label', 'Fear_Label', 'Threat_Label', 'Negative_Affect_Label', 'Urgency_Label']
Thresholds: {'Distress_Label': 0.24, 'Fear_Label': 0.5, 'Threat_Label': 0.45, 'Negative_Affect_Label': 0.29, 'Urgency_Label': 0.72}


In [2]:
def predict_medha(text):
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Move tensors to GPU/CPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = medha_model(**inputs)

    probabilities = torch.sigmoid(outputs.logits)[0].cpu().numpy()

    results = {}

    for i, label in enumerate(LABELS):
        threshold = THRESHOLDS[label]
        probability = float(probabilities[i])

        results[label] = {
            "probability": probability,
            "threshold": threshold,
            "prediction": int(probability >= threshold)
        }

    return results

In [3]:
def show_medha_prediction(text):
    
    results = predict_medha(text)

    print("=" * 70)
    print("MEDHA PREDICTION")
    print("=" * 70)
    
    print("\nTEXT:")
    print(text)
    
    print("\nRESULTS:")
    print("-" * 70)

    for label, result in results.items():
        
        status = "YES" if result["prediction"] == 1 else "NO"

        print(
            f"{label:<25} "
            f"Probability: {result['probability']:.3f} | "
            f"Threshold: {result['threshold']:.2f} | "
            f"Prediction: {status}"
        )

    print("=" * 70)

In [4]:
show_medha_prediction(
    "I feel calm today and things are going well."
)

MEDHA PREDICTION

TEXT:
I feel calm today and things are going well.

RESULTS:
----------------------------------------------------------------------
Distress_Label            Probability: 0.002 | Threshold: 0.24 | Prediction: NO
Fear_Label                Probability: 0.002 | Threshold: 0.50 | Prediction: NO
Threat_Label              Probability: 0.002 | Threshold: 0.45 | Prediction: NO
Negative_Affect_Label     Probability: 0.002 | Threshold: 0.29 | Prediction: NO
Urgency_Label             Probability: 0.001 | Threshold: 0.72 | Prediction: NO


In [5]:
show_medha_prediction(
    "I am scared about what might happen to my family."
)

MEDHA PREDICTION

TEXT:
I am scared about what might happen to my family.

RESULTS:
----------------------------------------------------------------------
Distress_Label            Probability: 0.023 | Threshold: 0.24 | Prediction: NO
Fear_Label                Probability: 0.997 | Threshold: 0.50 | Prediction: YES
Threat_Label              Probability: 0.025 | Threshold: 0.45 | Prediction: NO
Negative_Affect_Label     Probability: 0.019 | Threshold: 0.29 | Prediction: NO
Urgency_Label             Probability: 0.023 | Threshold: 0.72 | Prediction: NO


In [6]:
show_medha_prediction(
    "Mujhe darr lag raha hai aur tension ho rahi hai ki mere parivar ke saath kya hoga."
)

MEDHA PREDICTION

TEXT:
Mujhe darr lag raha hai aur tension ho rahi hai ki mere parivar ke saath kya hoga.

RESULTS:
----------------------------------------------------------------------
Distress_Label            Probability: 0.003 | Threshold: 0.24 | Prediction: NO
Fear_Label                Probability: 0.989 | Threshold: 0.50 | Prediction: YES
Threat_Label              Probability: 0.617 | Threshold: 0.45 | Prediction: YES
Negative_Affect_Label     Probability: 0.016 | Threshold: 0.29 | Prediction: NO
Urgency_Label             Probability: 0.006 | Threshold: 0.72 | Prediction: NO


In [7]:
show_medha_prediction(
    "Mujhe darr lag raha hai aur tension ho rahi hai ki mere parivar ke saath kya hoga."
)

MEDHA PREDICTION

TEXT:
Mujhe darr lag raha hai aur tension ho rahi hai ki mere parivar ke saath kya hoga.

RESULTS:
----------------------------------------------------------------------
Distress_Label            Probability: 0.003 | Threshold: 0.24 | Prediction: NO
Fear_Label                Probability: 0.989 | Threshold: 0.50 | Prediction: YES
Threat_Label              Probability: 0.617 | Threshold: 0.45 | Prediction: YES
Negative_Affect_Label     Probability: 0.016 | Threshold: 0.29 | Prediction: NO
Urgency_Label             Probability: 0.006 | Threshold: 0.72 | Prediction: NO


In [8]:
show_medha_prediction(
    "मुझे डर लग रहा है कि आगे क्या होगा और मैं अपने परिवार को लेकर चिंतित हूँ।"
)

MEDHA PREDICTION

TEXT:
मुझे डर लग रहा है कि आगे क्या होगा और मैं अपने परिवार को लेकर चिंतित हूँ।

RESULTS:
----------------------------------------------------------------------
Distress_Label            Probability: 0.012 | Threshold: 0.24 | Prediction: NO
Fear_Label                Probability: 0.995 | Threshold: 0.50 | Prediction: YES
Threat_Label              Probability: 0.025 | Threshold: 0.45 | Prediction: NO
Negative_Affect_Label     Probability: 0.010 | Threshold: 0.29 | Prediction: NO
Urgency_Label             Probability: 0.017 | Threshold: 0.72 | Prediction: NO


In [9]:
def get_medha_text_features(text):
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = medha_model(**inputs)

    probabilities = torch.sigmoid(outputs.logits)[0].cpu().numpy()

    return probabilities

In [10]:
features = get_medha_text_features(
    "Mujhe darr lag raha hai aur tension ho rahi hai ki mere parivar ke saath kya hoga."
)

print(features)
print("Shape:", features.shape)

[0.00273025 0.98852855 0.61718094 0.01569019 0.00573379]
Shape: (5,)


In [11]:
features = get_medha_text_features(
    "Mujhe darr lag raha hai aur tension ho rahi hai ki mere parivar ke saath kya hoga."
)

print(features)
print("Shape:", features.shape)

[0.00273025 0.98852855 0.61718094 0.01569019 0.00573379]
Shape: (5,)
